In [1]:
import pandas as pd
from difflib import SequenceMatcher
import random

In [2]:
combined_annotations = pd.read_csv("F:\\Berkeley\\Sp26\\INFO159\\AP\\combined_annotations.tsv", sep="\t", index_col=0)

In [52]:
combined_annotations.index[0]

'10014'

In [5]:
mismatch_count = 0
adjuticated = pd.DataFrame(columns=["ID", "Label", "Text"])
id, text, label = [], [], []
for i in range(499):
    if type(combined_annotations.iloc[2*i, 1]) != str:
        id.append(0) # Manually adjudicated ID as it was a non-string entry
        text.append("") # Empty text for non-string entry
        label.append("No Propaganda") # Label as "No Propaganda" for non-string entry
        continue
    if type(combined_annotations.iloc[2*i+1, 1]) != str:
        id.append(0) # Manually adjudicated ID as it was a non-string entry
        text.append("") # Empty text for non-string entry
        label.append("No Propaganda") # Label as "No Propaganda" for non-string entry
        continue
    s = SequenceMatcher(None, combined_annotations.iloc[2*i, 1], combined_annotations.iloc[2*i+1, 1])
    if s.ratio() == 1.0:
        id.append(combined_annotations.index[2*i]) # Use the ID of the first annotation for adjudicated data
        text.append(combined_annotations.iloc[2*i, 0]) # Use the text from the first annotation
        label.append(combined_annotations.iloc[2*i, 1]) # Use the label from the first annotation
    else:
        mismatch_count += 1
        if any(x in [combined_annotations.iloc[2*i, 1], combined_annotations.iloc[2*i+1, 1]] for x in ["No Propaganda", "Loaded Language"]) and any(x in [combined_annotations.iloc[2*i, 1], combined_annotations.iloc[2*i+1, 1]] for x in ["Appeal to Fear", "Name Calling"]):
            id.append(combined_annotations.index[2*i]) # Use the ID of the first annotation for adjudicated data
            text.append(combined_annotations.iloc[2*i, 0]) # Use the text from the first annotation
            label.append([x for x in [combined_annotations.iloc[2*i, 1], combined_annotations.iloc[2*i+1, 1]] if x != "No Propaganda" and x != "Loaded Language"][0]) # Label as not "No Propaganda" if either annotation is "No Propaganda"
        else:
            id.append(combined_annotations.index[2*i]) # Use the ID of the first annotation for adjudicated data
            text.append(combined_annotations.iloc[2*i, 0]) # Use the text from the first annotation
            if random.random() < 0.5:
                label.append(combined_annotations.iloc[2*i, 1]) # Randomly choose one of the two annotations
            else:
                label.append(combined_annotations.iloc[2*i+1, 1]) # Randomly choose one of the two annotations
#        print(f"Mismatch at index {2*i} and {2*i+1}: {combined_annotations.iloc[2*i, 1]} vs {combined_annotations.iloc[2*i+1, 1]}")

id.append(0) # Manually adjudicated ID for the last entry as it was a non-string entry
text.append("") # Empty text for the last entry
label.append("No Propaganda") # Label as "No Propaganda" for the last entry

adjuticated["ID"] = id
adjuticated["Text"] = text
adjuticated["Label"] = label
adjuticated

,ID,Label,Text
0,10014,Appeal to Fear,Â Safeguarding Womenâ€™s Sports: As an athlete...
1,10097,Appeal to Fear,57 KB) DEFENSE ACCESS ROADS III Recipient: Dep...
2,10126,No Propaganda,""" By augmenting its marked vehicle fleet, the ..."
3,10143,No Propaganda,", Tamuning, Guam, 96913Amount: $1,993,772.40Ju..."
4,10171,No Propaganda,Congressman Jackson has been working to build ...
...,...,...,...
495,9670,No Propaganda,"As a Member of Congress, Congresswoman Wilson ..."
496,9705,Loaded Language,"Today, over half of the girls have been releas..."
497,9734,Loaded Language,Our current system is broken. With backlogs la...
498,9988,No Propaganda,"So far, much of my work on the Foreign Affairs..."


In [60]:
adjuticated.to_csv("F:\\Berkeley\\Sp26\\INFO159\\AP\\adjuticated.tsv", sep="\t", index=False)

In [6]:
train = adjuticated.sample(frac=0.6, random_state=42, replace=False)
adjuticated.drop(train.index, inplace=True)
adjuticated.reset_index(drop=True, inplace=True)
print(len(train), len(adjuticated))
test = adjuticated.sample(frac=0.5, random_state=42, replace=False)
adjuticated.drop(test.index, inplace=True)
adjuticated.reset_index(drop=True, inplace=True)
print(len(test), len(adjuticated))
dev = adjuticated.sample(frac=1.0, random_state=42, replace=False)
adjuticated.drop(dev.index, inplace=True) # Clear the adjuticated DataFrame as all data has been split into train, test, and dev
print(len(dev), len(adjuticated))
train.to_csv("F:\\Berkeley\\Sp26\\INFO159\\AP\\AP5\\splits\\train.tsv", sep="\t", index=False)
test.to_csv("F:\\Berkeley\\Sp26\\INFO159\\AP\\AP5\\splits\\test.tsv", sep="\t", index=False)
dev.to_csv("F:\\Berkeley\\Sp26\\INFO159\\AP\\AP5\\splits\\dev.tsv", sep="\t", index=False)

300 200
100 100
100 0
